In [ ]:
# =====================================================
# 🚀 PROJET LF 2026 — VERSION FINALE CORRIGÉE
# =====================================================

import pandas as pd
import numpy as np
import warnings
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

np.random.seed(42)
pio.templates.default = "plotly_white"
pd.set_option("display.float_format", lambda x: "{:.2f}".format(x))

print("🚀 Lancement projet LF 2026")

# =====================================================
# 1. DATA GENERATION
# =====================================================

dates = pd.date_range(start="2018-01-01", end="2025-12-31", freq="MS")
df = pd.DataFrame({"date": dates})
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

df = df.sort_values("date").reset_index(drop=True)

annual_recettes = {
    2018: 210, 2019: 225, 2020: 200, 2021: 235,
    2022: 240.5, 2023: 265, 2024: 282, 2025: 296
}

monthly_pattern = np.array([14.0,14.5,23.1,15.9,17.3,33.8,17.4,16.6,28.3,18.4,19.6,40.6])
seasonal_index = monthly_pattern / monthly_pattern.mean()

df["recettes_totales"] = 0.0
for y, target in annual_recettes.items():
    mask = df["year"] == y
    base = target / 12
    df.loc[mask, "recettes_totales"] = base * seasonal_index[df.loc[mask,"month"].values-1]

# Décomposition
df["IS"] = df["recettes_totales"] * 0.248
df["IR"] = df["recettes_totales"] * 0.120
df["TVA"] = df["recettes_totales"] * 0.262
df["douanes"] = df["recettes_totales"] * 0.047
df["autres"] = df["recettes_totales"] * 0.323

# Macro
annual_pib = {2018:1100,2019:1150,2020:1100,2021:1200,2022:1250,2023:1300,2024:1350,2025:1403}
df["PIB_mensuel"] = 0
for y,val in annual_pib.items():
    df.loc[df["year"]==y,"PIB_mensuel"] = val/12

df["inflation"] = 2.8 + np.sin(df.index/12*2*np.pi)*0.8 + np.random.normal(0,0.4,len(df))
df["taux_directeur"] = 3 + np.random.normal(0,0.25,len(df))
df["importations"] = df["PIB_mensuel"]*0.45 + np.random.normal(0,8,len(df))

df["depenses_totales"] = df["recettes_totales"]*1.53
df["solde_budgetaire"] = df["recettes_totales"] - df["depenses_totales"]

# =====================================================
# 2. FEATURES + ADF
# =====================================================

print("ADF p-value:", adfuller(df["recettes_totales"])[1])

for lag in [1,3,6,12]:
    df[f"recettes_lag{lag}"] = df["recettes_totales"].shift(lag)
    df[f"PIB_lag{lag}"] = df["PIB_mensuel"].shift(lag)

df["inflation_lag1"] = df["inflation"].shift(1)
df["taux_directeur_lag2"] = df["taux_directeur"].shift(2)
df["import_lag1"] = df["importations"].shift(1)

df_model = df.dropna().copy()
df_model = pd.get_dummies(df_model, columns=["month"], drop_first=True)

# =====================================================
# 3. MODELS
# =====================================================

train = df_model[df_model["date"] < "2024-01-01"]
test = df_model[df_model["date"] >= "2024-01-01"]

y_train = train["recettes_totales"]
y_test = test["recettes_totales"]

exog_cols = ["PIB_lag1","PIB_lag3","inflation_lag1","taux_directeur_lag2","import_lag1"]
exog_train = train[exog_cols]
exog_test = test[exog_cols]

features = exog_cols + [c for c in df_model.columns if c.startswith("month_")]
X_train = train[features]
X_test = test[features]

# SARIMA (optimized search)
best_aic = np.inf
for p in range(2):
    for d in range(2):
        for q in range(2):
            for P in range(2):
                for D in range(2):
                    for Q in range(2):
                        try:
                            m = SARIMAX(y_train, order=(p,d,q), seasonal_order=(P,D,Q,12))
                            r = m.fit(disp=False)
                            if r.aic < best_aic:
                                best_aic = r.aic
                                best_order = (p,d,q)
                                best_seasonal = (P,D,Q,12)
                        except:
                            pass

sarima = SARIMAX(y_train, order=best_order, seasonal_order=best_seasonal).fit(disp=False)
sarimax = SARIMAX(y_train, exog=exog_train, order=best_order, seasonal_order=best_seasonal).fit(disp=False)

# ML models
lr = LinearRegression().fit(X_train, y_train)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsRegressor(5).fit(X_train_scaled, y_train)
rf = RandomForestRegressor(200, random_state=42).fit(X_train, y_train)
gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, random_state=42).fit(X_train, y_train)

# =====================================================
# 4. EVALUATION
# =====================================================

def eval_model(y_true,y_pred,name):
    return {
        "Modèle":name,
        "RMSE":np.sqrt(mean_squared_error(y_true,y_pred)),
        "MAE":mean_absolute_error(y_true,y_pred),
        "MAPE (%)":np.mean(np.abs((y_true-y_pred)/y_true))*100,
        "R²":r2_score(y_true,y_pred)
    }

preds = {
    "SARIMA": sarima.get_prediction(start=len(train), end=len(train)+len(test)-1).predicted_mean,
    "SARIMAX": sarimax.get_prediction(start=len(train), end=len(train)+len(test)-1, exog=exog_test).predicted_mean,
    "LR": lr.predict(X_test),
    "KNN": knn.predict(X_test_scaled),
    "RF": rf.predict(X_test),
    "GB": gb.predict(X_test)
}

metrics = pd.DataFrame([eval_model(y_test,preds[k],k) for k in preds])
print(metrics.round(3))

# =====================================================
# 5. FORECAST 2026
# =====================================================

future_dates = pd.date_range("2026-01-01","2026-12-31",freq="MS")

scenarios = {
    "BASE": {"growth":1.035,"target":390},
    "OPT": {"growth":1.045,"target":420},
    "PESS": {"growth":1.02,"target":360}
}

predictions = {}

for name,p in scenarios.items():
    pib = 1403 * p["growth"] / 12
    exog_future = pd.DataFrame({
        "PIB_lag1":[pib]*12,
        "PIB_lag3":[pib]*12,
        "inflation_lag1":[2.5]*12,
        "taux_directeur_lag2":[2.75]*12,
        "import_lag1":[60]*12
    })

    forecast = sarimax.get_forecast(steps=12, exog=exog_future)
    pred = forecast.predicted_mean.values

    pred *= p["target"] / pred.sum()
    predictions[name] = pred

# =====================================================
# 6. DASHBOARD
# =====================================================

fig = make_subplots(rows=2, cols=2)

# Historique
fig.add_trace(go.Scatter(x=df["date"], y=df["recettes_totales"], name="Historique"),1,1)

# Forecast
for k,v in predictions.items():
    fig.add_trace(go.Scatter(x=future_dates, y=v, name=k),1,2)

# Performance
fig.add_trace(go.Bar(x=metrics["Modèle"], y=metrics["MAPE (%)"]),2,1)

# Solde
base = predictions["BASE"]
fig.add_trace(go.Scatter(x=future_dates, y=base-base*1.53, name="Solde"),2,2)

fig.update_layout(height=800, title="LF 2026 Dashboard", hovermode="x unified")
fig.show()

# =====================================================
# 7. EXPORT
# =====================================================

fig.write_html("dashboard.html")

pd.DataFrame(predictions).to_csv("forecast_2026.csv")

print("✅ DONE")


🚀 Lancement projet LF 2026
ADF p-value: 0.981196887173741
    Modèle  RMSE  MAE  MAPE (%)   R²
0   SARIMA  1.61 1.19      3.84 0.97
1  SARIMAX  1.63 1.31      9.56 0.97
2       LR  1.79 1.25      4.68 0.96
3      KNN  5.91 4.86     18.24 0.56
4       RF  3.94 3.23     12.19 0.81
5       GB  3.83 3.11     12.00 0.82


✅ DONE
